In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path

def load_environment_variables():
    """
    Loads the environment variables from .env.
    Loads OpenAI api key and target model name used in the experiment
    New environment variables as necessary will be appended
    """

    PROJECT_ROOT = Path.cwd().parent
    ENV_FILE = PROJECT_ROOT / ".env"

    if ENV_FILE.exists():
        print(
            f".env file located at: {ENV_FILE.resolve()}")
        
        load_dotenv()
        env = {
                "api_key":              os.getenv("OPENAI_API_KEY"),
                "model_name":           os.getenv("OPENAI_MODEL_NAME"),
                "input_token_price":    float(os.getenv('MODEL_INPUT_TOKEN_PRICE')),
                "output_token_price":   float(os.getenv('MODEL_OUTPUT_TOKEN_PRICE'))
            }

    else:
        print(
            f".env not found at: {ENV_FILE.resolve()}"
        )
        env = {
            "api_key":              None,
            "model_name":           None,
            "input_token_price":    None,
            "output_token_price":   None
        }

    return env

In [2]:
from openai import OpenAI

env = load_environment_variables()

client = OpenAI(
    api_key=env['api_key']
)

response = client.chat.completions.create(
    model=env['model_name'],
    messages=[
        {
            "role": "user",
            "content": "what is RAG?"
        }
    ]
)

print(response.choices[0].message.content)

NameError: name 'Path' is not defined

In [ ]:
response = client.chat.completions.create(
    model=env['model_name'],
    messages=[
        {
            "role": "user",
            "content": "what is RAG in Natural Language Processing?"
        }
    ]
)

print(response.choices[0].message.content)

2026-11-02 09:51:17,469 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In Natural Language Processing (NLP), **RAG** stands for **Retrieval-Augmented Generation**. It is a framework that combines **retrieval-based methods** with **generative models** to improve the quality and factual accuracy of generated text.

### How RAG works:
1. **Retrieval Step:** Given an input query or context, a retrieval module searches a large external knowledge base (such as a document corpus or database) to find relevant documents or passages.
2. **Augmentation Step:** The retrieved documents are then used as additional context or evidence.
3. **Generation Step:** A generative model (typically a sequence-to-sequence language model like BART or T5) conditions on both the original input and the retrieved information to produce a final output, such as an answer to a question or a piece of generated text.

### Why use RAG?
- **Improved accuracy:** By grounding generation in retrieved knowledge, the model is less likely to hallucinate or produce factually incorrect content.
- **S

In [ ]:
questions = [
    "what is overfitting in machine learning?",
    "Explain gradient descent in simple terms.",
    "what is neural network?",
    "Difference between supervised and unsupervised learning?",
    "Explain backpropagation in neural networks."
]

In [ ]:
PROMT_V1 = """
Answer the following student questions clearly.
Question: {Question}
"""

In [ ]:
import time
import pandas as pd

def evaluate_prompt(
        questions,
        prompt_template,
        prompt_version,
        model,
        input_token_price,
        output_token_price
):
    results = []

    for idx, question in enumerate(questions, start=1):

        prompt = prompt_template.format(
            Question=question
        )
        
        start_time = time.time()

        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        end_time = time.time()

        latency = end_time - start_time

        # token cost calculation
        input_token_price = env['input_token_price']
        output_token_price = env['output_token_price']

        input_cost = (
            response.usage.prompt_tokens / (1024 * 1024) 
        ) * input_token_price

        output_cost = (
            response.usage.completion_tokens / (1024 * 1024)
        ) * output_token_price

        total_cost = input_cost + output_cost
        
        results.append(
            {
                "prompt": prompt,
                "response": response.choices[0].message.content,
                "question_id": idx,
                "prompt_version": prompt_version,
                "model": "gpt-4.1-mini",
                "latency_seconds": latency,
                "input_tokens": response.usage.prompt_tokens,
                "output_tokens": response.usage.completion_tokens,
                "total_tokens": response.usage.total_tokens,
                "input_token_price": input_cost,
                "output_token_price": output_cost,
                "total_cost": total_cost,
                "response_status": "success"


            }
        )

    return pd.DataFrame(results)

In [ ]:
df_v1 = evaluate_prompt(
    questions,
    PROMT_V1,
    "v1",
    env['model_name'],
    env['input_token_price'],
    env['output_token_price']
)

print("Statistics of V1 evaluation")
print(df_v1['latency_seconds'].describe())

latency_stats_v1 = df_v1['latency_seconds'].agg(
    ['mean', 'median', "min", 'max', 'std']
).to_frame("value")

print("Selected parameters")
print(latency_stats_v1)

for _, row in df_v1.iterrows():
    print("PROMPT:")
    print(row["prompt"])

    print("\nRESPONSE:")
    print(row["response"])

    print("\n" + "="*80 + "\n")

2026-11-02 09:51:18,679 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 09:51:20,080 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 09:51:21,569 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 09:51:23,710 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 09:51:25,866 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Statistics of V1 evaluation
count    5.000000
mean     1.667878
std      0.461152
min      1.157428
25%      1.395448
50%      1.473099
75%      2.153492
max      2.159924
Name: latency_seconds, dtype: float64
Selected parameters
           value
mean    1.667878
median  1.473099
min     1.157428
max     2.159924
std     0.461152
PROMPT:

Answer the following student questions clearly.
Question: what is overfitting in machine learning?


RESPONSE:
Overfitting in machine learning happens when a model learns the training data too well, including its noise and random fluctuations, rather than just the underlying patterns. As a result, the model performs very well on the training data but poorly on new, unseen data because it hasn't generalized well. In other words, the model is too complex and tailored to the specifics of the training set, which reduces its ability to make accurate predictions on other data.


PROMPT:

Answer the following student questions clearly.
Question: Explain grad

In [ ]:
def add_cost_metrics(df, env):
    """
    Function to calculate and add cost metric to input data frame.
    The method assumes that the dataframe contains input and output token price.
    """
    input_token_price = env['input_token_price']
    output_token_price = env['output_token_price']

    df['input_cost'] = (
        df['input_tokens'] / (1024 * 1024) 
    ) * input_token_price

    df['output_cost'] = (
        df['output_tokens'] / (1024 * 1024)
    ) * output_token_price

    df['total_cost'] = df['input_cost'] + df['output_cost']

    return df

In [ ]:
#df_v1 = add_cost_metrics(df=df_v1, env=env)

df_v1.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   prompt              5 non-null      str    
 1   response            5 non-null      str    
 2   question_id         5 non-null      int64  
 3   prompt_version      5 non-null      str    
 4   model               5 non-null      str    
 5   latency_seconds     5 non-null      float64
 6   input_tokens        5 non-null      int64  
 7   output_tokens       5 non-null      int64  
 8   total_tokens        5 non-null      int64  
 9   input_token_price   5 non-null      float64
 10  output_token_price  5 non-null      float64
 11  total_cost          5 non-null      float64
 12  response_status     5 non-null      str    
dtypes: float64(4), int64(4), str(5)
memory usage: 5.0 KB


In [ ]:
# Average monthly cost
avg_cost = df_v1['total_cost'].mean()

print(f"Average cost per question: ${avg_cost:.4f}")

projected_monthly_cost = avg_cost * 10000

print(f"Project monthly average cost: ${projected_monthly_cost:.2f}", )

Average cost per question: $0.0001
Project monthly average cost: $0.89
